<a href="https://colab.research.google.com/github/NeuroNex-ModelX/drainage-failure-sri-lanka/blob/main/03_elevation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 · Land height per DS division (Open-Meteo elevation API, Copernicus DEM GLO-90)
**Owner:** Jithmi · **Needs:** notebook 00 · **Makes:** `data/processed/elevation_ds.csv`

Samples points every ~1 km inside each division and asks the API for their height
(100 points per request, ~600 calls in total, ~10–15 min). Progress is saved after every
division, so if Colab disconnects just re-run: it continues where it stopped.

In [ ]:
# --- SETUP ---
import os, sys, time
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['MODELX_BASE'] = '/content/drive/MyDrive/ModelX'
sys.path.append(os.path.join(os.environ.get('MODELX_BASE', '..'), 'src'))
from common import *
import requests
import geopandas as gpd
from shapely.geometry import Point
from shapely.prepared import prep

STEP_DEG = 0.02     # ~1.1 km grid
API = 'https://api.open-meteo.com/v1/elevation'
ds_poly = gpd.read_file(os.path.join(PROC, 'ds_divisions.gpkg'))
partial = os.path.join(PROC, 'elevation_points_partial.csv')
done = pd.read_csv(partial) if os.path.exists(partial) else pd.DataFrame(columns=['pcode', 'lat', 'lon', 'elev'])
done_codes = set(done['pcode'])
print(f'{len(done_codes)} of {len(ds_poly)} divisions already done')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
105 of 339 divisions already done


In [ ]:
STEP_DEG = 0.02      # ~2 km grid: 4x fewer API calls than 0.01, enough for mean / min / % below 10 m


class DailyLimit(Exception):
    pass


def get_heights(points, tries=8):
    """points: list of (lat, lon) -> list of heights (m). Waits politely when rate-limited (HTTP 429)."""
    params = {'latitude': ','.join(f'{a:.4f}' for a, _ in points),
              'longitude': ','.join(f'{b:.4f}' for _, b in points)}
    for k in range(tries):
        r = requests.get(API, params=params, timeout=60)
        if r.status_code == 200:
            return r.json()['elevation']
        reason = r.text.lower()
        if r.status_code == 429 and 'daily' in reason:
            raise DailyLimit(r.text[:200])
        wait = 3700 if 'hourly' in reason else 65      # hourly limit -> wait an hour, otherwise a minute
        print(f'   rate limited ({r.status_code}) - waiting {wait // 60} min...', flush=True)
        time.sleep(wait)
    raise RuntimeError(f'API failed: {r.status_code} {r.text[:200]}')


try:
    for _, row in ds_poly.iterrows():
        if row['pcode'] in done_codes:
            continue
        geom = row.geometry
        pg = prep(geom)
        minx, miny, maxx, maxy = geom.bounds
        xs, ys = np.meshgrid(np.arange(minx, maxx, STEP_DEG), np.arange(miny, maxy, STEP_DEG))
        pts = [(y, x) for x, y in zip(xs.ravel(), ys.ravel()) if pg.contains(Point(x, y))]
        if len(pts) < 5:   # very small division: add a point inside it
            rp = geom.representative_point()
            pts = pts + [(rp.y, rp.x)]
        heights = []
        for i in range(0, len(pts), 100):
            heights += get_heights(pts[i:i + 100])
            time.sleep(1.0)                            # gentle pace: ~60 requests a minute
        chunk = pd.DataFrame({'pcode': row['pcode'], 'lat': [p[0] for p in pts],
                              'lon': [p[1] for p in pts], 'elev': heights})
        chunk.to_csv(partial, mode='a', header=not os.path.exists(partial), index=False)
        print(row['pcode'], row['ds_name'], len(pts), 'points', flush=True)
    print('All divisions done.')
except DailyLimit as e:
    print('Daily API limit reached - progress is saved. Re-run the setup cell and this cell after '
          '5:30 am Sri Lanka time (the limit resets at midnight UTC).', e)

LK3209 Pasgoda 32 points
LK3212 Mulatiyana 24 points
LK3215 Athuraliya 10 points
LK3218 Akuressa 28 points
LK3221 Welipitiya 15 points
LK3224 Malimbada 7 points
LK3227 Kaburupitiya 10 points
LK3230 Hakmana 7 points
LK3233 Kirinda Puhulwella 9 points
LK3236 Thihagoda 12 points
LK3239 Weligama 7 points
LK3242 Matara Four Gravets 12 points
LK3245 Devinuwara 8 points
LK3248 Dikwella 11 points
LK3303 Sooriyawewa 39 points
LK3306 Lunugamwehera 58 points
LK3309 Thissamaharama 164 points
LK3312 Hambantota 69 points
LK3315 Ambalantota 44 points
LK3318 Angunakolapelessa 34 points
LK3321 Weeraketiya 23 points
LK3324 Katuwana 20 points
LK3325 Walasmulla 20 points
LK3327 Okewela 9 points
LK3330 Beliatta 22 points
LK3333 Tangalle 31 points
LK4103 Islands North (Kayts) 6 points
LK4104 Karainagar 5 points
LK4106 Valikamam West (Chankanai) 10 points
LK4109 Valikamam South West (Sandilipay) 10 points
LK4112 Valikamam North (Thllippalai) 13 points
LK4115 Valikamam South (Uduvil) 6 points
LK4118 Valikamam

In [ ]:
pts = pd.read_csv(partial)
pts.loc[pts['elev'] < -50, 'elev'] = np.nan   # sea / no-data
elev = pts.groupby('pcode')['elev'].agg(
    elev_mean='mean', elev_min='min', elev_median='median',
    pct_below_10m=lambda h: 100 * (h < 10).mean(),
    pct_below_5m=lambda h: 100 * (h < 5).mean(),
    n_points='size').reset_index()

ds_list = load_ds_list()[['pcode', 'is_coastal']]
elev = elev.merge(ds_list, on='pcode', how='right')
elev.to_csv(os.path.join(PROC, 'elevation_ds.csv'), index=False)
save_meta('elevation', 'Open-Meteo elevation API (Copernicus DEM GLO-90), ~1 km sample grid')
print(elev.describe().round(1))

       elev_mean  elev_min  elev_median  pct_below_10m  pct_below_5m  \
count      339.0     339.0        339.0          339.0         339.0   
mean       193.9      95.7        178.7           19.7           9.1   
std        312.1     191.6        300.9           31.4          18.0   
min          2.0       0.0          1.0            0.0           0.0   
25%         17.8       2.0         15.0            0.0           0.0   
50%         67.1      19.0         58.0            0.0           0.0   
75%        172.2      86.0        152.2           34.1          12.5   
max       1911.0    1334.0       1904.0          100.0         100.0   

       n_points  is_coastal  
count     339.0       339.0  
mean       61.7         0.3  
std        56.3         0.4  
min         1.0         0.0  
25%        22.0         0.0  
50%        42.0         0.0  
75%        82.0         1.0  
max       372.0         1.0  
